# Long-Term Deep Learning Experiment

This Colab notebook tests tabular deep learning models for the long-term NBA forecasting tasks.

The experiment keeps the existing long-term data design:

- source table: `data/gold/long_term_player_forecast_training.parquet`
- temporal split: train `<= 2019-20`, validation `2020-21`, test `2021-22`
- horizon targets: H1, H2, H3
- leakage control: future horizon columns and metadata columns are excluded from features

The goal is narrow: evaluate whether a tuned tabular MLP can beat the current tabular ML selections. LSTM is intentionally excluded because the current long-term dataset is season-anchor tabular data, not a clean sequence dataset.


In [ ]:
# Colab setup: mount Drive, install dependencies, and make project source importable.
from pathlib import Path
import subprocess
import sys

IN_COLAB = Path("/content").exists()
DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_PROJECT_DIR = DRIVE_ROOT / "nba-scout-assistant"
COLAB_REPO_DIR = Path("/content/nba-scout-assistant")
REPO_URL = "https://github.com/kdnehihi/nba-scout-assistant.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas",
        "pyarrow",
        "scikit-learn",
        "mlflow",
        "optuna",
        "lightgbm",
    ], check=True)

PROJECT_ROOT_CANDIDATES = [
    DRIVE_PROJECT_DIR,
    COLAB_REPO_DIR,
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve(),
]

PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if (path / "src").exists()), None)
if PROJECT_ROOT is None and IN_COLAB:
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    PROJECT_ROOT = COLAB_REPO_DIR
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project source directory containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = DRIVE_PROJECT_DIR / "data" if IN_COLAB else PROJECT_ROOT / "data"
OUTPUT_DIR = DRIVE_PROJECT_DIR / "reports" / "long_term_dl" if IN_COLAB else PROJECT_ROOT / "reports" / "long_term_dl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import random
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path

import mlflow
import numpy as np
try:
    import optuna
except ImportError:
    optuna = None
    print("Optuna is not installed. Falling back to a small manual MLP config search.")
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    brier_score_loss,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from src.dataset.loaders import load_long_term_training, resolve_data_paths

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


## Experiment Controls

`QUICK_MODE=True` is the default Colab setting. It keeps the search bounded while still testing useful MLP architectures. Increase trials or epochs only after confirming the notebook runs end to end.


In [ ]:
SEED = 42
QUICK_MODE = True
RUN_PER_100_TARGETS = True
LOG_FINAL_MODELS = False

LONG_TERM_HORIZONS = [1, 2, 3]
REGRESSION_TARGETS = ["pts_per_36", "ast_per_36", "reb_per_36"]
if RUN_PER_100_TARGETS:
    REGRESSION_TARGETS += ["pts_per_100", "ast_per_100", "reb_per_100"]
CLASSIFICATION_TARGETS = ["active_probability"]

MLP_OPTUNA_TRIALS = 6 if QUICK_MODE else 16
ML_OPTUNA_TRIALS_PER_FAMILY = 4 if QUICK_MODE else 12
MAX_EPOCHS = 60 if QUICK_MODE else 120
PATIENCE = 8 if QUICK_MODE else 12
MIN_DELTA = 1e-4
NUM_WORKERS = 0
N_JOBS = 1 if QUICK_MODE else -1

# Keep this as None to run all configured long-term tasks.
# Example override for a fast smoke test: [("pts_per_36", 1), ("active_probability", 1)]
TARGET_TASKS_OVERRIDE: list[tuple[str, int]] | None = None


def set_seed(seed: int = SEED) -> None:
    """Input: seed. Output: deterministic-ish numpy/random/torch state."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print({
    "quick_mode": QUICK_MODE,
    "mlp_optuna_trials": MLP_OPTUNA_TRIALS,
    "ml_optuna_trials_per_family": ML_OPTUNA_TRIALS_PER_FAMILY,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "run_per_100_targets": RUN_PER_100_TARGETS,
    "n_jobs": N_JOBS,
})


## Load Long-Term Gold Data

The notebook reads the same materialized long-term table used by notebook 01 and local dataset loaders.


In [ ]:
paths = resolve_data_paths(DATA_DIR)
long_term = load_long_term_training(paths)

print("long_term shape:", long_term.shape)
print("split counts:")
display(long_term["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))
print("anchor seasons by split:")
display(long_term.groupby("split")["anchor_season"].agg(lambda s: sorted(s.dropna().unique())).reset_index())
long_term.head()


## MLflow Setup

The tracking backend uses SQLite on Drive to avoid MLflow file-store limitations. The notebook logs the best MLP configuration and validation/test metrics for each target-horizon task.


In [ ]:
MLFLOW_DB_PATH = DRIVE_PROJECT_DIR / "mlflow.db" if IN_COLAB else PROJECT_ROOT / "mlflow.db"
MLFLOW_ARTIFACT_DIR = DRIVE_PROJECT_DIR / "mlartifacts" if IN_COLAB else PROJECT_ROOT / "mlartifacts"
MLFLOW_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")
mlflow.set_experiment("nba_scout_long_term_deep_learning")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow artifact URI:", MLFLOW_ARTIFACT_DIR)
print("MLflow experiment:", mlflow.get_experiment_by_name("nba_scout_long_term_deep_learning").name)


## Feature And Target Preparation

This section mirrors the existing long-term leakage rules. Future H1/H2/H3 columns are never allowed as features. Metadata columns are kept for audit but excluded from model input.


In [ ]:
LONG_TERM_METADATA_COLS = [
    "player_id",
    "player_name",
    "anchor_season",
    "anchor_season_start_year",
    "anchor_date",
    "team_id",
    "split",
]

LONG_TERM_TARGETS = {
    horizon: {
        "active_probability": f"active_h{horizon}",
        "pts_per_36": f"pts_per_36_h{horizon}",
        "ast_per_36": f"ast_per_36_h{horizon}",
        "reb_per_36": f"reb_per_36_h{horizon}",
        "pts_per_100": f"pts_per_100_h{horizon}",
        "ast_per_100": f"ast_per_100_h{horizon}",
        "reb_per_100": f"reb_per_100_h{horizon}",
    }
    for horizon in LONG_TERM_HORIZONS
}


def is_future_horizon_column(column: str) -> bool:
    """Input: column name. Output: True when it is an H1/H2/H3 future target or label."""
    return str(column).endswith(("_h1", "_h2", "_h3"))


def long_term_excluded_columns(horizon: int, columns: pd.Index | list[str]) -> set[str]:
    """Input: forecast horizon and dataframe columns. Output: columns excluded to prevent leakage."""
    excluded = set(LONG_TERM_METADATA_COLS)
    for target_group in LONG_TERM_TARGETS.values():
        excluded.update(target_group.values())
    excluded.update(column for column in columns if is_future_horizon_column(column))
    return excluded


def long_term_feature_columns(df: pd.DataFrame, horizon: int) -> list[str]:
    """Input: long-term dataframe and horizon. Output: model feature columns matching notebook 01 leakage rules."""
    excluded = long_term_excluded_columns(horizon, df.columns)
    feature_cols = []
    for column in df.columns:
        if column in excluded:
            continue
        if pd.api.types.is_datetime64_any_dtype(df[column]):
            continue
        if df[column].notna().sum() == 0:
            continue
        feature_cols.append(column)
    return feature_cols


def get_one_hot_encoder() -> OneHotEncoder:
    """Input: none. Output: sklearn OneHotEncoder compatible with the runtime version."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Input: raw feature matrix. Output: fitted-ready numeric/categorical preprocessing transformer."""
    categorical_cols = [column for column in X.columns if X[column].dtype == "object" or str(X[column].dtype) == "category"]
    numeric_cols = [column for column in X.columns if column not in categorical_cols]
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]), numeric_cols),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("one_hot", get_one_hot_encoder()),
            ]), categorical_cols),
        ],
        remainder="drop",
    )


def prepare_task_frame(df: pd.DataFrame, task_name: str, horizon: int) -> tuple[pd.DataFrame, pd.Series, pd.Series, list[str]]:
    """Input: long-term data and task. Output: raw X, y, split labels, and feature columns."""
    target_col = LONG_TERM_TARGETS[horizon][task_name]
    model_df = df.copy()
    if task_name != "active_probability":
        active_col = LONG_TERM_TARGETS[horizon]["active_probability"]
        model_df = model_df[model_df[active_col].eq(1)].copy()
    model_df = model_df.dropna(subset=[target_col]).copy()
    feature_cols = long_term_feature_columns(model_df, horizon)
    X = model_df[feature_cols].copy()
    y = pd.to_numeric(model_df[target_col], errors="coerce")
    splits = model_df["split"].copy()
    return X, y, splits, feature_cols


def configured_tasks() -> list[tuple[str, int]]:
    """Input: global settings. Output: target-horizon tasks to evaluate."""
    if TARGET_TASKS_OVERRIDE is not None:
        return TARGET_TASKS_OVERRIDE
    tasks = []
    for horizon in LONG_TERM_HORIZONS:
        for task in CLASSIFICATION_TARGETS + REGRESSION_TARGETS:
            if task in LONG_TERM_TARGETS[horizon]:
                tasks.append((task, horizon))
    return tasks


task_audit_rows = []
for task_name, horizon in configured_tasks():
    target_col = LONG_TERM_TARGETS[horizon][task_name]
    if target_col not in long_term.columns:
        continue
    X, y, splits, feature_cols = prepare_task_frame(long_term, task_name, horizon)
    task_audit_rows.append({
        "task": task_name,
        "horizon": horizon,
        "target_col": target_col,
        "rows": len(X),
        "train_rows": int(splits.eq("train").sum()),
        "validation_rows": int(splits.eq("validation").sum()),
        "test_rows": int(splits.eq("test").sum()),
        "feature_count": len(feature_cols),
        "target_missing_pct_after_filter": float(y.isna().mean()),
    })

task_audit = pd.DataFrame(task_audit_rows)
display(task_audit)


## Tabular MLP Implementation

The MLP is intentionally tabular. It receives the same feature matrix used by the current long-term ML models after imputation, scaling, and one-hot encoding.


In [ ]:
class TabularMLP(nn.Module):
    """Feed-forward neural network for tabular long-term forecasting."""

    def __init__(
        self,
        input_size: int,
        hidden_sizes: tuple[int, ...],
        output_size: int = 1,
        dropout: float = 0.10,
        batch_norm: bool = True,
    ):
        super().__init__()
        layers: list[nn.Module] = []
        current_size = input_size
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(current_size, hidden_size))
            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            current_size = hidden_size
        layers.append(nn.Linear(current_size, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Input: preprocessed tabular tensor. Output: raw prediction/logit."""
        return self.network(x).squeeze(1)


@dataclass(frozen=True)
class MLPConfig:
    hidden_sizes: tuple[int, ...]
    dropout: float
    learning_rate: float
    weight_decay: float
    batch_size: int
    loss_name: str
    batch_norm: bool


@dataclass
class PreparedTaskArrays:
    X_train: np.ndarray
    X_validation: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_validation: np.ndarray
    y_test: np.ndarray
    y_train_model: np.ndarray
    y_validation_model: np.ndarray
    y_test_model: np.ndarray
    target_mean: float
    target_std: float
    input_size: int
    feature_cols: list[str]
    train_rows: int
    validation_rows: int
    test_rows: int


def make_dataloader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    """Input: numpy arrays. Output: PyTorch dataloader."""
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS)


def prepare_arrays(X: pd.DataFrame, y: pd.Series, splits: pd.Series, task_type: str) -> PreparedTaskArrays:
    """Input: raw X/y/splits. Output: preprocessed train/validation/test arrays without split leakage."""
    train_mask = splits.eq("train")
    validation_mask = splits.eq("validation")
    test_mask = splits.eq("test")
    if not train_mask.any() or not validation_mask.any() or not test_mask.any():
        raise ValueError("Task requires train, validation, and test rows.")

    preprocessor = build_preprocessor(X)
    X_train = preprocessor.fit_transform(X[train_mask]).astype("float32")
    X_validation = preprocessor.transform(X[validation_mask]).astype("float32")
    X_test = preprocessor.transform(X[test_mask]).astype("float32")

    y_train = y[train_mask].to_numpy(dtype="float32")
    y_validation = y[validation_mask].to_numpy(dtype="float32")
    y_test = y[test_mask].to_numpy(dtype="float32")

    if task_type == "regression":
        target_mean = float(np.mean(y_train))
        target_std = float(np.std(y_train))
        if target_std == 0 or np.isnan(target_std):
            target_std = 1.0
        y_train_model = ((y_train - target_mean) / target_std).astype("float32")
        y_validation_model = ((y_validation - target_mean) / target_std).astype("float32")
        y_test_model = ((y_test - target_mean) / target_std).astype("float32")
    else:
        target_mean = 0.0
        target_std = 1.0
        y_train_model = y_train.astype("float32")
        y_validation_model = y_validation.astype("float32")
        y_test_model = y_test.astype("float32")

    return PreparedTaskArrays(
        X_train=X_train,
        X_validation=X_validation,
        X_test=X_test,
        y_train=y_train,
        y_validation=y_validation,
        y_test=y_test,
        y_train_model=y_train_model,
        y_validation_model=y_validation_model,
        y_test_model=y_test_model,
        target_mean=target_mean,
        target_std=target_std,
        input_size=X_train.shape[1],
        feature_cols=list(X.columns),
        train_rows=int(train_mask.sum()),
        validation_rows=int(validation_mask.sum()),
        test_rows=int(test_mask.sum()),
    )


def inverse_regression_target(values: np.ndarray, arrays: PreparedTaskArrays) -> np.ndarray:
    """Input: scaled regression predictions. Output: original target scale predictions."""
    return values * arrays.target_std + arrays.target_mean


## Training And Evaluation Helpers

Optuna selects configuration by validation MAE for regression and validation Brier score for classification. Test metrics are reported only after selecting the best validation configuration.


In [ ]:
def build_loss(task_type: str, config: MLPConfig, y_train_model: np.ndarray) -> nn.Module:
    """Input: task type and config. Output: PyTorch loss function."""
    if task_type == "classification":
        positive = float(np.sum(y_train_model == 1))
        negative = float(np.sum(y_train_model == 0))
        pos_weight = torch.tensor([negative / max(positive, 1.0)], dtype=torch.float32, device=DEVICE)
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    if config.loss_name == "mse":
        return nn.MSELoss()
    return nn.HuberLoss(delta=1.0)


def predict_raw(model: nn.Module, X: np.ndarray, batch_size: int) -> np.ndarray:
    """Input: fitted model and preprocessed matrix. Output: raw predictions/logits."""
    model.eval()
    preds = []
    loader = DataLoader(torch.tensor(X, dtype=torch.float32), batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for xb in loader:
            preds.append(model(xb.to(DEVICE)).detach().cpu().numpy())
    return np.concatenate(preds)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Input: actual and predicted regression targets. Output: MAE/RMSE/R2."""
    mse = mean_squared_error(y_true, y_pred)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mse)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def classification_metrics(y_true: np.ndarray, logits: np.ndarray) -> dict[str, float]:
    """Input: binary labels and logits. Output: Brier/ROC-AUC/F1 metrics."""
    probability = 1 / (1 + np.exp(-logits))
    prediction = (probability >= 0.5).astype(int)
    metrics = {
        "brier": float(brier_score_loss(y_true, probability)),
        "precision": float(precision_score(y_true, prediction, zero_division=0)),
        "recall": float(recall_score(y_true, prediction, zero_division=0)),
        "f1": float(f1_score(y_true, prediction, zero_division=0)),
    }
    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, probability))
    except ValueError:
        metrics["roc_auc"] = np.nan
    return metrics


def evaluate_mlp(
    model: nn.Module,
    arrays: PreparedTaskArrays,
    task_type: str,
    task_name: str,
    config: MLPConfig,
) -> pd.DataFrame:
    """Input: fitted model and arrays. Output: validation/test metric rows."""
    rows = []
    for split_name, X_split, y_split in [
        ("validation", arrays.X_validation, arrays.y_validation),
        ("test", arrays.X_test, arrays.y_test),
    ]:
        raw_pred = predict_raw(model, X_split, config.batch_size)
        if task_type == "classification":
            metrics = classification_metrics(y_split, raw_pred)
        else:
            y_pred = inverse_regression_target(raw_pred, arrays)
            if task_name.endswith("per_36") or task_name.endswith("per_100"):
                y_pred = np.clip(y_pred, 0, None)
            metrics = regression_metrics(y_split, y_pred)
        rows.append({"split": split_name, "rows": len(y_split), **metrics})
    return pd.DataFrame(rows)


def train_one_mlp(
    arrays: PreparedTaskArrays,
    task_type: str,
    config: MLPConfig,
    max_epochs: int = MAX_EPOCHS,
    patience: int = PATIENCE,
    min_delta: float = MIN_DELTA,
) -> tuple[TabularMLP, dict[str, float]]:
    """Input: task arrays and config. Output: best model and training diagnostics."""
    set_seed(SEED)
    model = TabularMLP(
        input_size=arrays.input_size,
        hidden_sizes=config.hidden_sizes,
        dropout=config.dropout,
        batch_norm=config.batch_norm,
    ).to(DEVICE)
    criterion = build_loss(task_type, config, arrays.y_train_model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    train_loader = make_dataloader(arrays.X_train, arrays.y_train_model, config.batch_size, shuffle=True)
    validation_loader = make_dataloader(arrays.X_validation, arrays.y_validation_model, config.batch_size, shuffle=False)

    best_state = None
    best_validation_loss = float("inf")
    stale_epochs = 0
    epochs_ran = 0

    for epoch in range(max_epochs):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        validation_losses = []
        with torch.no_grad():
            for xb, yb in validation_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                validation_losses.append(float(criterion(model(xb), yb).detach().cpu()))
        validation_loss = float(np.mean(validation_losses))
        epochs_ran = epoch + 1

        if validation_loss < best_validation_loss - min_delta:
            best_validation_loss = validation_loss
            best_state = deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    diagnostics = {
        "best_validation_loss": best_validation_loss,
        "epochs_ran": epochs_ran,
    }
    return model, diagnostics


## Optuna Search Space

The search space is intentionally compact. It tests shallow and medium MLPs, dropout, learning rate, weight decay, batch size, and Huber vs MSE for regression.


In [ ]:
HIDDEN_ARCHITECTURES = {
    "64": (64,),
    "128_64": (128, 64),
    "192_96": (192, 96),
    "256_128": (256, 128),
    "256_128_64": (256, 128, 64),
}


def mlp_config_from_params(params: dict[str, object], task_type: str) -> MLPConfig:
    """Input: Optuna params. Output: concrete MLP config."""
    hidden_key = str(params["hidden_architecture"])
    return MLPConfig(
        hidden_sizes=HIDDEN_ARCHITECTURES[hidden_key],
        dropout=float(params["dropout"]),
        learning_rate=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"]),
        batch_size=int(params["batch_size"]),
        loss_name="bce" if task_type == "classification" else str(params["loss_name"]),
        batch_norm=bool(params["batch_norm"]),
    )


def suggest_mlp_config(trial, task_type: str) -> MLPConfig:
    """Input: Optuna trial and task type. Output: candidate MLP config."""
    params = {
        "hidden_architecture": trial.suggest_categorical("hidden_architecture", list(HIDDEN_ARCHITECTURES)),
        "dropout": trial.suggest_float("dropout", 0.05, 0.35),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 3e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
        "batch_norm": trial.suggest_categorical("batch_norm", [True, False]),
    }
    if task_type != "classification":
        params["loss_name"] = trial.suggest_categorical("loss_name", ["huber", "mse"])
    return mlp_config_from_params(params, task_type)


def validation_metric_for_config(
    arrays: PreparedTaskArrays,
    task_type: str,
    task_name: str,
    config: MLPConfig,
) -> tuple[float, dict[str, float]]:
    """Input: arrays and config. Output: validation selection metric and training diagnostics."""
    model, diagnostics = train_one_mlp(arrays, task_type, config)
    raw_pred = predict_raw(model, arrays.X_validation, config.batch_size)
    if task_type == "classification":
        metric = classification_metrics(arrays.y_validation, raw_pred)["brier"]
    else:
        y_pred = inverse_regression_target(raw_pred, arrays)
        if task_name.endswith("per_36") or task_name.endswith("per_100"):
            y_pred = np.clip(y_pred, 0, None)
        metric = regression_metrics(arrays.y_validation, y_pred)["mae"]
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return float(metric), diagnostics


def objective_factory(arrays: PreparedTaskArrays, task_type: str, task_name: str):
    """Input: task arrays and task metadata. Output: Optuna objective function."""
    def objective(trial) -> float:
        config = suggest_mlp_config(trial, task_type)
        metric, diagnostics = validation_metric_for_config(arrays, task_type, task_name, config)
        trial.set_user_attr("epochs_ran", diagnostics["epochs_ran"])
        trial.set_user_attr("best_validation_loss", diagnostics["best_validation_loss"])
        return metric
    return objective


def fallback_config_grid(task_type: str) -> list[MLPConfig]:
    """Input: task type. Output: compact manual configs used when Optuna is unavailable."""
    loss_names = ["bce"] if task_type == "classification" else ["huber", "mse"]
    configs = []
    for hidden_sizes, dropout, learning_rate, weight_decay, batch_size, batch_norm in [
        ((64,), 0.10, 1e-3, 1e-4, 128, True),
        ((128, 64), 0.15, 7e-4, 3e-4, 128, True),
        ((256, 128), 0.20, 5e-4, 1e-4, 256, True),
        ((256, 128, 64), 0.25, 5e-4, 5e-4, 256, False),
    ]:
        for loss_name in loss_names:
            configs.append(MLPConfig(hidden_sizes, dropout, learning_rate, weight_decay, batch_size, loss_name, batch_norm))
    return configs


def run_task_search(task_name: str, horizon: int) -> tuple[TabularMLP, pd.DataFrame, pd.DataFrame, dict[str, object]]:
    """Input: task and horizon. Output: best model, evaluation rows, trial table, selected config metadata."""
    task_type = "classification" if task_name == "active_probability" else "regression"
    X, y, splits, feature_cols = prepare_task_frame(long_term, task_name, horizon)
    arrays = prepare_arrays(X, y, splits, task_type)

    if optuna is not None:
        sampler = optuna.samplers.TPESampler(seed=SEED)
        study = optuna.create_study(direction="minimize", sampler=sampler, study_name=f"mlp_{task_name}_h{horizon}")
        study.optimize(objective_factory(arrays, task_type, task_name), n_trials=MLP_OPTUNA_TRIALS, show_progress_bar=False)
        best_config = mlp_config_from_params(study.best_trial.params, task_type)
        best_trial_value = float(study.best_value)
        trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
    else:
        trial_rows = []
        best_config = None
        best_trial_value = float("inf")
        for number, config in enumerate(fallback_config_grid(task_type)[:MLP_OPTUNA_TRIALS]):
            metric, diagnostics = validation_metric_for_config(arrays, task_type, task_name, config)
            trial_rows.append({
                "number": number,
                "value": metric,
                "params_hidden_sizes": str(config.hidden_sizes),
                "params_dropout": config.dropout,
                "params_learning_rate": config.learning_rate,
                "params_weight_decay": config.weight_decay,
                "params_batch_size": config.batch_size,
                "params_loss_name": config.loss_name,
                "params_batch_norm": config.batch_norm,
                "user_attrs_epochs_ran": diagnostics["epochs_ran"],
                "user_attrs_best_validation_loss": diagnostics["best_validation_loss"],
                "state": "COMPLETE",
            })
            if metric < best_trial_value:
                best_trial_value = metric
                best_config = config
        trials = pd.DataFrame(trial_rows)
        if best_config is None:
            raise ValueError("No fallback MLP config was evaluated.")

    best_model, diagnostics = train_one_mlp(
        arrays=arrays,
        task_type=task_type,
        config=best_config,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
    )
    evaluation = evaluate_mlp(best_model, arrays, task_type, task_name, best_config)
    evaluation.insert(0, "model", "tabular_mlp_optuna")
    evaluation.insert(1, "task", task_name)
    evaluation.insert(2, "horizon", horizon)
    evaluation.insert(3, "target_col", LONG_TERM_TARGETS[horizon][task_name])
    evaluation.insert(4, "feature_count", len(feature_cols))
    evaluation["features"] = ", ".join(feature_cols)
    evaluation["best_trial_value"] = best_trial_value
    evaluation["epochs_ran"] = diagnostics["epochs_ran"]
    evaluation["best_validation_loss"] = diagnostics["best_validation_loss"]

    trials.insert(0, "task", task_name)
    trials.insert(1, "horizon", horizon)

    config_metadata = {
        "task": task_name,
        "horizon": horizon,
        "task_type": task_type,
        "target_col": LONG_TERM_TARGETS[horizon][task_name],
        "feature_count": len(feature_cols),
        "features": feature_cols,
        "best_config": best_config.__dict__,
        "best_trial_value": best_trial_value,
        "train_rows": arrays.train_rows,
        "validation_rows": arrays.validation_rows,
        "test_rows": arrays.test_rows,
        **diagnostics,
    }
    return best_model, evaluation, trials, config_metadata


## Run Long-Term MLP Experiments

This cell runs Optuna tuning per target-horizon task, trains the selected MLP again, evaluates validation/test, and logs results to MLflow.


In [ ]:
all_evaluations = []
all_trials = []
all_configs = []
trained_models = {}

for task_name, horizon in configured_tasks():
    target_col = LONG_TERM_TARGETS[horizon].get(task_name)
    if target_col is None or target_col not in long_term.columns:
        print(f"Skipping {task_name} h{horizon}: missing target column")
        continue

    print(f"Running MLP search for {task_name} h{horizon} ({target_col})")
    try:
        model, evaluation, trials, config_metadata = run_task_search(task_name, horizon)
    except Exception as exc:
        print(f"FAILED {task_name} h{horizon}: {exc}")
        continue

    trained_models[f"mlp_{task_name}_h{horizon}"] = model
    all_evaluations.append(evaluation)
    all_trials.append(trials)
    all_configs.append(config_metadata)

    with mlflow.start_run(run_name=f"long_term_mlp_{task_name}_h{horizon}"):
        mlflow.set_tags({
            "project": "nba-scout-assistant",
            "experiment_type": "long_term_tabular_mlp",
            "task": task_name,
            "horizon": horizon,
        })
        mlflow.log_params({
            "task": task_name,
            "horizon": horizon,
            "target_col": config_metadata["target_col"],
            "task_type": config_metadata["task_type"],
            "feature_count": config_metadata["feature_count"],
            "optuna_trials": MLP_OPTUNA_TRIALS,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "seed": SEED,
            **{f"best_{key}": value for key, value in config_metadata["best_config"].items()},
        })
        mlflow.log_metric("best_trial_value", float(config_metadata["best_trial_value"]))
        mlflow.log_metric("epochs_ran", float(config_metadata["epochs_ran"]))
        mlflow.log_metric("best_validation_loss", float(config_metadata["best_validation_loss"]))
        for _, row in evaluation.iterrows():
            prefix = row["split"]
            for metric in ["rows", "mae", "rmse", "r2", "brier", "roc_auc", "precision", "recall", "f1"]:
                if metric in row and pd.notna(row[metric]):
                    mlflow.log_metric(f"{prefix}_{metric}", float(row[metric]))
        if LOG_FINAL_MODELS:
            mlflow.pytorch.log_model(model, name="model")

    display(evaluation)

long_term_mlp_evaluation = pd.concat(all_evaluations, ignore_index=True) if all_evaluations else pd.DataFrame()
long_term_mlp_trials = pd.concat(all_trials, ignore_index=True) if all_trials else pd.DataFrame()
long_term_mlp_configs = pd.DataFrame(all_configs)

print("evaluation rows:", long_term_mlp_evaluation.shape)
display(long_term_mlp_evaluation.sort_values(["task", "horizon", "split"]))


## Tune Tabular ML Baselines With Optuna

This section tunes the non-deep-learning baselines with the same feature preparation and temporal split. Each model family receives a small bounded search budget in quick mode, then the best family is selected by validation metric.


In [ ]:
def try_import_lightgbm_models():
    """Input: none. Output: optional LightGBM classifier/regressor classes."""
    try:
        from lightgbm import LGBMClassifier, LGBMRegressor  # type: ignore
        return LGBMClassifier, LGBMRegressor
    except Exception as exc:
        print("LightGBM unavailable; skipping LightGBM trials:", exc)
        return None, None


def available_ml_families(task_type: str) -> list[str]:
    """Input: task type. Output: model families to tune for that task."""
    if task_type == "classification":
        families = ["logistic", "random_forest", "hist_gradient_boosting"]
        LGBMClassifier, _ = try_import_lightgbm_models()
        if LGBMClassifier is not None:
            families.append("lightgbm")
        return families
    families = ["ridge", "random_forest", "hist_gradient_boosting"]
    _, LGBMRegressor = try_import_lightgbm_models()
    if LGBMRegressor is not None:
        families.append("lightgbm")
    return families


def build_tuned_ml_model(family: str, task_type: str, X: pd.DataFrame, params: dict[str, object]) -> Pipeline:
    """Input: model family, task type, raw X, and params. Output: sklearn pipeline."""
    preprocessor = build_preprocessor(X)

    if task_type == "classification":
        if family == "logistic":
            return Pipeline([
                ("preprocess", preprocessor),
                ("model", LogisticRegression(
                    max_iter=2500,
                    class_weight="balanced",
                    C=float(params.get("C", 0.5)),
                    random_state=SEED,
                )),
            ])
        if family == "random_forest":
            return Pipeline([
                ("preprocess", preprocessor),
                ("model", RandomForestClassifier(
                    n_estimators=int(params.get("n_estimators", 300)),
                    min_samples_leaf=int(params.get("min_samples_leaf", 8)),
                    max_features=params.get("max_features", "sqrt"),
                    class_weight="balanced",
                    n_jobs=N_JOBS,
                    random_state=SEED,
                )),
            ])
        if family == "hist_gradient_boosting":
            return Pipeline([
                ("preprocess", preprocessor),
                ("model", HistGradientBoostingClassifier(
                    max_iter=int(params.get("max_iter", 200)),
                    learning_rate=float(params.get("learning_rate", 0.04)),
                    l2_regularization=float(params.get("l2_regularization", 0.05)),
                    min_samples_leaf=int(params.get("min_samples_leaf", 20)),
                    random_state=SEED,
                )),
            ])
        if family == "lightgbm":
            LGBMClassifier, _ = try_import_lightgbm_models()
            if LGBMClassifier is None:
                raise ImportError("LightGBM classifier is unavailable")
            return Pipeline([
                ("preprocess", preprocessor),
                ("model", LGBMClassifier(
                    n_estimators=int(params.get("n_estimators", 250)),
                    learning_rate=float(params.get("learning_rate", 0.03)),
                    num_leaves=int(params.get("num_leaves", 31)),
                    min_child_samples=int(params.get("min_child_samples", 20)),
                    subsample=float(params.get("subsample", 0.9)),
                    colsample_bytree=float(params.get("colsample_bytree", 0.9)),
                    reg_alpha=float(params.get("reg_alpha", 0.0)),
                    reg_lambda=float(params.get("reg_lambda", 0.0)),
                    class_weight="balanced",
                    random_state=SEED,
                    verbosity=-1,
                )),
            ])

    if family == "ridge":
        return Pipeline([
            ("preprocess", preprocessor),
            ("model", Ridge(alpha=float(params.get("alpha", 5.0)))),
        ])
    if family == "random_forest":
        return Pipeline([
            ("preprocess", preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=int(params.get("n_estimators", 300)),
                min_samples_leaf=int(params.get("min_samples_leaf", 8)),
                max_features=params.get("max_features", "sqrt"),
                n_jobs=N_JOBS,
                random_state=SEED,
            )),
        ])
    if family == "hist_gradient_boosting":
        return Pipeline([
            ("preprocess", preprocessor),
            ("model", HistGradientBoostingRegressor(
                max_iter=int(params.get("max_iter", 250)),
                learning_rate=float(params.get("learning_rate", 0.04)),
                l2_regularization=float(params.get("l2_regularization", 0.05)),
                min_samples_leaf=int(params.get("min_samples_leaf", 20)),
                random_state=SEED,
            )),
        ])
    if family == "lightgbm":
        _, LGBMRegressor = try_import_lightgbm_models()
        if LGBMRegressor is None:
            raise ImportError("LightGBM regressor is unavailable")
        return Pipeline([
            ("preprocess", preprocessor),
            ("model", LGBMRegressor(
                n_estimators=int(params.get("n_estimators", 350)),
                learning_rate=float(params.get("learning_rate", 0.03)),
                num_leaves=int(params.get("num_leaves", 31)),
                min_child_samples=int(params.get("min_child_samples", 20)),
                subsample=float(params.get("subsample", 0.9)),
                colsample_bytree=float(params.get("colsample_bytree", 0.9)),
                reg_alpha=float(params.get("reg_alpha", 0.0)),
                reg_lambda=float(params.get("reg_lambda", 0.0)),
                random_state=SEED,
                verbosity=-1,
            )),
        ])
    raise ValueError(f"Unsupported ML family: {family}")


def suggest_ml_params(trial, family: str, task_type: str) -> dict[str, object]:
    """Input: Optuna trial, model family, and task type. Output: candidate hyperparameters."""
    if family == "logistic":
        return {"C": trial.suggest_float("C", 0.05, 5.0, log=True)}
    if family == "ridge":
        return {"alpha": trial.suggest_float("alpha", 0.1, 50.0, log=True)}
    if family == "random_forest":
        return {
            "n_estimators": trial.suggest_categorical("n_estimators", [200, 300, 500]),
            "min_samples_leaf": trial.suggest_categorical("min_samples_leaf", [4, 8, 12, 20]),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.45, 0.65, 0.85]),
        }
    if family == "hist_gradient_boosting":
        return {
            "max_iter": trial.suggest_categorical("max_iter", [150, 250, 350]),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
            "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 0.5, log=True),
            "min_samples_leaf": trial.suggest_categorical("min_samples_leaf", [10, 20, 35, 50]),
        }
    if family == "lightgbm":
        return {
            "n_estimators": trial.suggest_categorical("n_estimators", [200, 350, 500]),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
            "num_leaves": trial.suggest_categorical("num_leaves", [15, 31, 47]),
            "min_child_samples": trial.suggest_categorical("min_child_samples", [15, 25, 40, 60]),
            "subsample": trial.suggest_float("subsample", 0.75, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 0.5, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 1.0, log=True),
        }
    raise ValueError(f"Unsupported ML family: {family}")


def fallback_ml_param_grid(family: str) -> list[dict[str, object]]:
    """Input: model family. Output: compact manual params when Optuna is unavailable."""
    if family == "logistic":
        return [{"C": value} for value in [0.25, 0.5, 1.0, 2.0]]
    if family == "ridge":
        return [{"alpha": value} for value in [1.0, 5.0, 10.0, 25.0]]
    if family == "random_forest":
        return [
            {"n_estimators": 200, "min_samples_leaf": 8, "max_features": "sqrt"},
            {"n_estimators": 300, "min_samples_leaf": 8, "max_features": 0.65},
            {"n_estimators": 300, "min_samples_leaf": 12, "max_features": "sqrt"},
            {"n_estimators": 500, "min_samples_leaf": 12, "max_features": 0.45},
        ]
    if family == "hist_gradient_boosting":
        return [
            {"max_iter": 150, "learning_rate": 0.05, "l2_regularization": 0.05, "min_samples_leaf": 20},
            {"max_iter": 250, "learning_rate": 0.04, "l2_regularization": 0.05, "min_samples_leaf": 20},
            {"max_iter": 350, "learning_rate": 0.025, "l2_regularization": 0.10, "min_samples_leaf": 35},
            {"max_iter": 250, "learning_rate": 0.03, "l2_regularization": 0.01, "min_samples_leaf": 10},
        ]
    if family == "lightgbm":
        return [
            {"n_estimators": 250, "learning_rate": 0.03, "num_leaves": 31, "min_child_samples": 25, "subsample": 0.9, "colsample_bytree": 0.9, "reg_alpha": 0.01, "reg_lambda": 0.1},
            {"n_estimators": 500, "learning_rate": 0.025, "num_leaves": 31, "min_child_samples": 40, "subsample": 0.9, "colsample_bytree": 0.8, "reg_alpha": 0.05, "reg_lambda": 0.25},
        ]
    return [{}]


def evaluate_sklearn_model(model: Pipeline, X: pd.DataFrame, y: pd.Series, splits: pd.Series, task_type: str) -> pd.DataFrame:
    """Input: fitted sklearn model and split data. Output: validation/test metrics."""
    rows = []
    for split_name in ["validation", "test"]:
        mask = splits.eq(split_name)
        if not mask.any():
            continue
        y_true = y[mask].to_numpy(dtype="float64")
        if task_type == "classification":
            probability = model.predict_proba(X[mask])[:, 1]
            logits = np.log(np.clip(probability, 1e-6, 1 - 1e-6) / np.clip(1 - probability, 1e-6, 1 - 1e-6))
            metrics = classification_metrics(y_true, logits)
        else:
            y_pred = np.asarray(model.predict(X[mask]), dtype="float64")
            y_pred = np.clip(y_pred, 0, None)
            metrics = regression_metrics(y_true, y_pred)
        rows.append({"split": split_name, "rows": int(mask.sum()), **metrics})
    return pd.DataFrame(rows)


def tune_ml_family(task_name: str, horizon: int, family: str) -> tuple[Pipeline, pd.DataFrame, pd.DataFrame, dict[str, object]]:
    """Input: task, horizon, and model family. Output: best model, metrics, trials, and config metadata."""
    task_type = "classification" if task_name == "active_probability" else "regression"
    X, y, splits, feature_cols = prepare_task_frame(long_term, task_name, horizon)
    train_mask = splits.eq("train")
    validation_mask = splits.eq("validation")
    selection_metric = "brier" if task_type == "classification" else "mae"

    def score_params(params: dict[str, object]) -> tuple[float, Pipeline]:
        model = build_tuned_ml_model(family, task_type, X, params)
        model.fit(X[train_mask], y[train_mask])
        validation_metrics = evaluate_sklearn_model(model, X[validation_mask | train_mask | splits.eq("test")], y[validation_mask | train_mask | splits.eq("test")], splits[validation_mask | train_mask | splits.eq("test")], task_type)
        row = validation_metrics[validation_metrics["split"].eq("validation")]
        if row.empty:
            raise ValueError("No validation metrics were produced.")
        return float(row.iloc[0][selection_metric]), model

    trial_rows = []
    best_value = float("inf")
    best_params: dict[str, object] | None = None
    best_model: Pipeline | None = None

    if optuna is not None:
        sampler = optuna.samplers.TPESampler(seed=SEED)
        study = optuna.create_study(direction="minimize", sampler=sampler, study_name=f"ml_{family}_{task_name}_h{horizon}")

        def objective(trial) -> float:
            params = suggest_ml_params(trial, family, task_type)
            value, _ = score_params(params)
            return value

        study.optimize(objective, n_trials=ML_OPTUNA_TRIALS_PER_FAMILY, show_progress_bar=False)
        best_params = dict(study.best_trial.params)
        best_value, best_model = score_params(best_params)
        trials = study.trials_dataframe(attrs=("number", "value", "params", "state"))
    else:
        for number, params in enumerate(fallback_ml_param_grid(family)[:ML_OPTUNA_TRIALS_PER_FAMILY]):
            value, model = score_params(params)
            trial_rows.append({"number": number, "value": value, "params": params, "state": "COMPLETE"})
            if value < best_value:
                best_value = value
                best_params = params
                best_model = model
        trials = pd.DataFrame(trial_rows)

    if best_model is None or best_params is None:
        raise ValueError(f"No model selected for {family} {task_name} h{horizon}")

    evaluation = evaluate_sklearn_model(best_model, X, y, splits, task_type)
    evaluation.insert(0, "model", f"tuned_{family}")
    evaluation.insert(1, "task", task_name)
    evaluation.insert(2, "horizon", horizon)
    evaluation.insert(3, "target_col", LONG_TERM_TARGETS[horizon][task_name])
    evaluation.insert(4, "feature_count", len(feature_cols))
    evaluation["features"] = ", ".join(feature_cols)
    evaluation["best_trial_value"] = best_value

    trials.insert(0, "family", family)
    trials.insert(0, "horizon", horizon)
    trials.insert(0, "task", task_name)

    config_metadata = {
        "task": task_name,
        "horizon": horizon,
        "task_type": task_type,
        "family": family,
        "target_col": LONG_TERM_TARGETS[horizon][task_name],
        "feature_count": len(feature_cols),
        "features": feature_cols,
        "best_params": best_params,
        "best_trial_value": best_value,
        "selection_metric": selection_metric,
        "train_rows": int(train_mask.sum()),
        "validation_rows": int(splits.eq("validation").sum()),
        "test_rows": int(splits.eq("test").sum()),
    }
    return best_model, evaluation, trials, config_metadata


def tune_ml_task(task_name: str, horizon: int) -> tuple[dict[str, Pipeline], pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Input: task and horizon. Output: family models, evaluations, trials, and selected-family summary."""
    task_type = "classification" if task_name == "active_probability" else "regression"
    metric = "brier" if task_type == "classification" else "mae"
    models = {}
    evaluations = []
    trials = []
    configs = []

    for family in available_ml_families(task_type):
        print(f"Tuning ML {family} for {task_name} h{horizon}")
        try:
            model, evaluation, family_trials, config = tune_ml_family(task_name, horizon, family)
        except Exception as exc:
            print(f"FAILED ML {family} {task_name} h{horizon}: {exc}")
            continue
        models[f"{family}_{task_name}_h{horizon}"] = model
        evaluations.append(evaluation)
        trials.append(family_trials)
        configs.append(config)

    if not evaluations:
        return models, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    evaluation_df = pd.concat(evaluations, ignore_index=True)
    trials_df = pd.concat(trials, ignore_index=True) if trials else pd.DataFrame()
    config_df = pd.DataFrame(configs)
    validation = evaluation_df[evaluation_df["split"].eq("validation")].copy()
    best_validation = validation.sort_values(metric).groupby(["task", "horizon"], as_index=False).first()
    selected_models = set(best_validation["model"])
    selected_eval = evaluation_df[evaluation_df["model"].isin(selected_models)].copy()
    selected_eval["selected_by"] = f"validation_{metric}"
    return models, evaluation_df, trials_df, selected_eval


## Run Tuned ML Baseline Experiments

This cell tunes the classical/tabular ML candidates per task and horizon. The result is a fairer comparator against the tuned MLP than the fixed notebook-01 settings.


In [ ]:
all_ml_evaluations = []
all_ml_trials = []
all_ml_selected = []
tuned_ml_models = {}

for task_name, horizon in configured_tasks():
    target_col = LONG_TERM_TARGETS[horizon].get(task_name)
    if target_col is None or target_col not in long_term.columns:
        print(f"Skipping tuned ML {task_name} h{horizon}: missing target column")
        continue

    print(f"Running tuned ML search for {task_name} h{horizon} ({target_col})")
    models, evaluation, trials, selected_eval = tune_ml_task(task_name, horizon)
    tuned_ml_models.update(models)
    if not evaluation.empty:
        all_ml_evaluations.append(evaluation)
    if not trials.empty:
        all_ml_trials.append(trials)
    if not selected_eval.empty:
        all_ml_selected.append(selected_eval)

    with mlflow.start_run(run_name=f"long_term_tuned_ml_{task_name}_h{horizon}"):
        mlflow.set_tags({
            "project": "nba-scout-assistant",
            "experiment_type": "long_term_tuned_tabular_ml",
            "task": task_name,
            "horizon": horizon,
        })
        mlflow.log_params({
            "task": task_name,
            "horizon": horizon,
            "target_col": target_col,
            "trials_per_family": ML_OPTUNA_TRIALS_PER_FAMILY,
            "seed": SEED,
        })
        if not selected_eval.empty:
            for _, row in selected_eval.iterrows():
                prefix = f"selected_{row['split']}"
                mlflow.log_param("selected_model", row["model"])
                for metric in ["rows", "mae", "rmse", "r2", "brier", "roc_auc", "precision", "recall", "f1"]:
                    if metric in row and pd.notna(row[metric]):
                        mlflow.log_metric(f"{prefix}_{metric}", float(row[metric]))

    display(selected_eval)

long_term_tuned_ml_evaluation = pd.concat(all_ml_evaluations, ignore_index=True) if all_ml_evaluations else pd.DataFrame()
long_term_tuned_ml_trials = pd.concat(all_ml_trials, ignore_index=True) if all_ml_trials else pd.DataFrame()
long_term_tuned_ml_selected = pd.concat(all_ml_selected, ignore_index=True) if all_ml_selected else pd.DataFrame()

print("tuned ML evaluation rows:", long_term_tuned_ml_evaluation.shape)
display(long_term_tuned_ml_selected.sort_values(["task", "horizon", "split"]))


## Tuned MLP Versus Tuned ML

This is the main fair comparison in this notebook. The winner is selected by validation metric, with test reported as holdout evidence.


In [ ]:
def compare_tuned_mlp_vs_tuned_ml(mlp_eval: pd.DataFrame, ml_eval: pd.DataFrame) -> pd.DataFrame:
    """Input: tuned MLP and tuned ML metrics. Output: task-level comparison by split."""
    if mlp_eval.empty or ml_eval.empty:
        return pd.DataFrame()
    rows = []
    for row in mlp_eval.itertuples(index=False):
        metric = "brier" if row.task == "active_probability" else "mae"
        candidates = ml_eval[
            ml_eval["task"].eq(row.task)
            & ml_eval["horizon"].eq(row.horizon)
            & ml_eval["split"].eq(row.split)
        ].copy()
        if candidates.empty or metric not in candidates.columns:
            continue
        best_ml = candidates.sort_values(metric).iloc[0]
        mlp_value = getattr(row, metric)
        rows.append({
            "task": row.task,
            "horizon": row.horizon,
            "split": row.split,
            "metric": metric,
            "mlp_value": mlp_value,
            "validation_selected_tuned_ml_model": best_ml["model"],
            "validation_selected_tuned_ml_value": best_ml[metric],
            "mlp_minus_validation_selected_tuned_ml": mlp_value - best_ml[metric],
            "mlp_wins": mlp_value < best_ml[metric],
        })
    return pd.DataFrame(rows)


long_term_mlp_vs_tuned_ml = compare_tuned_mlp_vs_tuned_ml(long_term_mlp_evaluation, long_term_tuned_ml_selected)
display(long_term_mlp_vs_tuned_ml.sort_values(["task", "horizon", "split"]))

if not long_term_mlp_vs_tuned_ml.empty:
    promote_summary = (
        long_term_mlp_vs_tuned_ml
        .pivot_table(index=["task", "horizon"], columns="split", values="mlp_wins", aggfunc="first")
        .reset_index()
    )
    promote_summary["mlp_wins_validation_and_test"] = promote_summary.get("validation", False).fillna(False) & promote_summary.get("test", False).fillna(False)
    display(promote_summary.sort_values(["task", "horizon"]))


## Compare Against Existing Long-Term ML Metrics

If notebook 01 already saved `long_term_model_evaluation.parquet`, this section compares MLP against the current selected/baseline tabular ML results.


In [ ]:
EXISTING_LONG_TERM_EVAL_PATH = paths.gold_dir / "long_term_model_evaluation.parquet"

if EXISTING_LONG_TERM_EVAL_PATH.exists() and not long_term_mlp_evaluation.empty:
    existing_eval = pd.read_parquet(EXISTING_LONG_TERM_EVAL_PATH)
    existing_subset = existing_eval[existing_eval["split"].isin(["validation", "test"])].copy()
    existing_subset = existing_subset.rename(columns={"target": "task"})
    comparison_rows = []
    for row in long_term_mlp_evaluation.itertuples(index=False):
        task_existing = existing_subset[
            existing_subset["task"].eq(row.task)
            & existing_subset["horizon"].eq(row.horizon)
            & existing_subset["split"].eq(row.split)
        ].copy()
        if task_existing.empty:
            continue
        metric = "brier" if row.task == "active_probability" else "mae"
        if metric not in task_existing.columns or metric not in long_term_mlp_evaluation.columns:
            continue
        best_existing = task_existing.sort_values(metric).iloc[0]
        mlp_value = getattr(row, metric)
        comparison_rows.append({
            "task": row.task,
            "horizon": row.horizon,
            "split": row.split,
            "metric": metric,
            "mlp_value": mlp_value,
            "best_existing_model": best_existing.get("model"),
            "best_existing_value": best_existing[metric],
            "mlp_minus_existing": mlp_value - best_existing[metric],
            "mlp_wins": mlp_value < best_existing[metric],
        })
    long_term_mlp_vs_existing = pd.DataFrame(comparison_rows)
    display(long_term_mlp_vs_existing.sort_values(["task", "horizon", "split"]))
else:
    long_term_mlp_vs_existing = pd.DataFrame()
    print("Existing long-term model evaluation file not found or MLP evaluation is empty:", EXISTING_LONG_TERM_EVAL_PATH)


## Save Experiment Outputs

These outputs can be copied into the project decision notes after reviewing validation/test stability.


In [ ]:
EVAL_OUTPUT_PATH = OUTPUT_DIR / "long_term_mlp_evaluation.parquet"
TRIALS_OUTPUT_PATH = OUTPUT_DIR / "long_term_mlp_optuna_trials.parquet"
CONFIG_OUTPUT_PATH = OUTPUT_DIR / "long_term_mlp_selected_configs.json"
COMPARISON_OUTPUT_PATH = OUTPUT_DIR / "long_term_mlp_vs_existing.parquet"
TUNED_ML_EVAL_OUTPUT_PATH = OUTPUT_DIR / "long_term_tuned_ml_evaluation.parquet"
TUNED_ML_TRIALS_OUTPUT_PATH = OUTPUT_DIR / "long_term_tuned_ml_optuna_trials.parquet"
TUNED_ML_SELECTED_OUTPUT_PATH = OUTPUT_DIR / "long_term_tuned_ml_selected.parquet"
TUNED_COMPARISON_OUTPUT_PATH = OUTPUT_DIR / "long_term_mlp_vs_tuned_ml.parquet"

if not long_term_mlp_evaluation.empty:
    long_term_mlp_evaluation.to_parquet(EVAL_OUTPUT_PATH, index=False)
    print("Saved", EVAL_OUTPUT_PATH)
if not long_term_mlp_trials.empty:
    long_term_mlp_trials.to_parquet(TRIALS_OUTPUT_PATH, index=False)
    print("Saved", TRIALS_OUTPUT_PATH)
if all_configs:
    CONFIG_OUTPUT_PATH.write_text(json.dumps(all_configs, indent=2, default=str))
    print("Saved", CONFIG_OUTPUT_PATH)
if not long_term_mlp_vs_existing.empty:
    long_term_mlp_vs_existing.to_parquet(COMPARISON_OUTPUT_PATH, index=False)
    print("Saved", COMPARISON_OUTPUT_PATH)

if "long_term_tuned_ml_evaluation" in globals() and not long_term_tuned_ml_evaluation.empty:
    long_term_tuned_ml_evaluation.to_parquet(TUNED_ML_EVAL_OUTPUT_PATH, index=False)
    print("Saved", TUNED_ML_EVAL_OUTPUT_PATH)
if "long_term_tuned_ml_trials" in globals() and not long_term_tuned_ml_trials.empty:
    long_term_tuned_ml_trials.to_parquet(TUNED_ML_TRIALS_OUTPUT_PATH, index=False)
    print("Saved", TUNED_ML_TRIALS_OUTPUT_PATH)
if "long_term_tuned_ml_selected" in globals() and not long_term_tuned_ml_selected.empty:
    long_term_tuned_ml_selected.to_parquet(TUNED_ML_SELECTED_OUTPUT_PATH, index=False)
    print("Saved", TUNED_ML_SELECTED_OUTPUT_PATH)
if "long_term_mlp_vs_tuned_ml" in globals() and not long_term_mlp_vs_tuned_ml.empty:
    long_term_mlp_vs_tuned_ml.to_parquet(TUNED_COMPARISON_OUTPUT_PATH, index=False)
    print("Saved", TUNED_COMPARISON_OUTPUT_PATH)


## Reading Guide

Use validation metrics for model selection and test metrics only as the holdout check.

For regression targets, lower `mae` is better. For `active_probability`, lower `brier` is better and higher `roc_auc` is better.

Promote MLP only when it beats the current selected ML model on validation and does not show a clear test degradation. If gains are small or inconsistent, keep the current tabular ML models and document MLP as tested but not selected.
